# TrashScan — Path B3 (5 classes)

Este notebook documenta e executa o experimento **Path B3**, a variante do Path B que combina um detector YOLO previamente treinado no Path A com um classificador **ViT-B/16 pré-treinado em ImageNet21k e ajustado em ImageNet-1k**. O objetivo é avaliar se um classificador visual forte, treinado sobre recortes de objetos detectados, melhora a classificação final das 5 classes grosseiras do TrashScan: `plastic`, `paper`, `metal`, `glass` e `other`.

## Metodologia do Path B3

O Path B é um pipeline em duas etapas. Primeiro, um detector YOLO localiza os objetos nas imagens completas. Depois, cada região detectada é recortada, redimensionada para 224x224 e enviada para um classificador de imagem. No B3, esse classificador é o `vit_b16_imagenet`, implementado no script de treino como `vit_base_patch16_224.augreg_in21k_ft_in1k` via `timm`.

Neste notebook, o detector não é treinado novamente. Em vez disso, as runs existentes do Path A são varridas, o melhor `weights/best.pt` é selecionado com base em métricas de detecção, e esse peso é usado de forma congelada para gerar os crops do Path B. O classificador ViT-B/16 é então fine-tuned sobre esses crops, usando validação para escolher o melhor checkpoint.

A avaliação considera dois níveis de resultado: métricas do classificador isolado sobre crops e uma avaliação combinada detector + classificador, que se aproxima mais do uso final do sistema em imagens completas.

## Estrutura esperada no RunPod

Este notebook assume que a estrutura **já está baixada/criada no volume** do RunPod:

- `/workspace/.venv`
- `/workspace/TrashScan`
- `/workspace/TACO`
- `/workspace/external_datasets`
- `/workspace/processed_5cls`
- `/workspace/runs`

O foco aqui é rodar o fluxo do **Path B** usando os scripts `.py` do projeto, sem refazer downloads, merge ou preprocessamento por padrão. As células abaixo seguem a ordem experimental: preparar ambiente, validar entradas, escolher o detector do Path A, treinar o B3, resumir métricas e executar a avaliação final combinada.

## 1) Imports, paths e utilitários

Os imports reúnem as bibliotecas usadas no notebook, enquanto os caminhos centralizam os diretórios importantes do RunPod. A função auxiliar `run_cmd` padroniza a execução dos scripts do projeto e imprime o comando antes de rodá-lo, o que facilita a reprodução do experimento.

A organização dos caminhos importa porque o Path B3 depende de artefatos produzidos antes deste notebook: o dataset processado em `/workspace/processed_5cls` e os pesos do detector treinado no Path A em `/workspace/runs`. A célula também cria as pastas de saída do Path B e imprime os caminhos, funcionando como uma primeira checagem visual do ambiente.

In [1]:
from pathlib import Path
import sys
import subprocess

VENV_DIR = Path("/workspace/.venv")
REQS_PATH = Path("/workspace/TrashScan/env/environment.txt")

python_bin = VENV_DIR / "bin" / "python"
pip_bin = VENV_DIR / "bin" / "pip"

if not VENV_DIR.exists():
    print("Creating venv at", VENV_DIR)
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
else:
    print("Venv already exists at", VENV_DIR)

if not python_bin.exists():
    raise FileNotFoundError(f"python not found at {python_bin}")

if not pip_bin.exists():
    raise FileNotFoundError(f"pip not found at {pip_bin}")

subprocess.run([str(pip_bin), "install", "--upgrade", "pip"], check=True)

if REQS_PATH.exists():
    subprocess.run([str(pip_bin), "install", "-r", str(REQS_PATH)], check=True)
else:
    raise FileNotFoundError(f"requirements file not found: {REQS_PATH}")

subprocess.run([str(pip_bin), "install", "ipykernel"], check=True)
subprocess.run(
    [
        str(python_bin),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        "trashscan-venv",
        "--display-name",
        "TrashScan (.venv)",
    ],
    check=True,
)

print("Kernel registered: TrashScan (.venv)")
print("Switch the notebook kernel to use:", python_bin)

Venv already exists at /workspace/.venv


Installed kernelspec trashscan-venv in /root/.local/share/jupyter/kernels/trashscan-venv
Kernel registered: TrashScan (.venv)
Switch the notebook kernel to use: /workspace/.venv/bin/python


In [15]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil
import torch
import pandas as pd
import json

# Caminhos principais no RunPod
WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls'

DATASET_YAML_PATH_A = PROCESSED_DIR / 'dataset_path_A.yaml'
DATASET_YAML_PATH_B = PROCESSED_DIR / 'dataset_path_B.yaml'

# Path A: necessário porque o Path B usa o melhor detector treinado no Path A
RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A'

# Path B3: treino do classificador ViT-B/16 ImageNet21k
RUNS_PATH_B_DIR = WORKSPACE / 'runs' / 'path_B_tta_wbf'
MLFLOW_DIR = Path('/root/mlflow')

# Resultados finais persistentes no volume
RESULTS_PATH_B_DIR = WORKSPACE / 'results_path_B'

# Scripts principais
TRAIN_PATH_B_SCRIPT = TRAIN_DIR / 'train_path_B.py'

for p in [RUNS_PATH_B_DIR, MLFLOW_DIR, RESULTS_PATH_B_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('TACO_DIR           =', TACO_DIR)
print('EXTERNAL_DIR       =', EXTERNAL_DIR)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_A=', DATASET_YAML_PATH_A)
print('DATASET_YAML_PATH_B=', DATASET_YAML_PATH_B)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('RUNS_PATH_B_DIR    =', RUNS_PATH_B_DIR)
print('MLFLOW_DIR         =', MLFLOW_DIR)
print('RESULTS_PATH_B_DIR =', RESULTS_PATH_B_DIR)
print('TRAIN_PATH_B_SCRIPT=', TRAIN_PATH_B_SCRIPT)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    """Roda comandos de forma previsível no notebook."""
    if isinstance(cmd, str):
        print('$', cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)
    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)

Python             = /workspace/.venv/bin/python
REPO_ROOT          = /workspace/TrashScan
TACO_DIR           = /workspace/TACO
EXTERNAL_DIR       = /workspace/external_datasets
PROCESSED_DIR      = /workspace/processed_5cls
DATASET_YAML_PATH_A= /workspace/processed_5cls/dataset_path_A.yaml
DATASET_YAML_PATH_B= /workspace/processed_5cls/dataset_path_B.yaml
RUNS_PATH_A_DIR    = /workspace/runs/path_A
RUNS_PATH_B_DIR    = /workspace/runs/path_B_tta_wbf
MLFLOW_DIR         = /root/mlflow
RESULTS_PATH_B_DIR = /workspace/results_path_B
TRAIN_PATH_B_SCRIPT= /workspace/TrashScan/train/paths/train_path_B.py


## 2) Verificação dos arquivos principais

Antes de iniciar a seleção de pesos ou o treino, esta checagem confirma se existem o script `train_path_B.py`, o diretório processado de 5 classes e ao menos a pasta-base de runs do Path A.

Se algo estiver ausente, o notebook lista explicitamente o caminho faltante. Isso evita gastar tempo depurando erros posteriores de treino que, na prática, seriam apenas consequência de arquivos não montados ou preprocessamento não executado.

In [12]:
required_paths = [
    TRAIN_PATH_B_SCRIPT,
    PROCESSED_DIR,
    RUNS_PATH_A_DIR,
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Arquivos/pastas ausentes:")
    for p in missing:
        print(" -", p)
else:
    print("Tudo certo.")

Tudo certo.


## 3) Configuração de GPU e hiperparâmetros do B3

A escolha do dispositivo é feita automaticamente: GPU `0` quando CUDA está disponível, ou CPU caso contrário. Em seguida, ficam definidos os principais hiperparâmetros do fine-tuning.

Para o B3, o classificador selecionado é apenas `vit_b16_imagenet`. O treino é configurado para até 100 épocas, batch size 8, learning rate `5e-5`, early stopping com paciência 10 e limiar de confiança `0.25` para o detector YOLO usado na geração dos crops.

In [7]:
if torch.cuda.is_available():
    DEVICE = "0"
    gpu_name = torch.cuda.get_device_properties(0).name
else:
    DEVICE = "cpu"
    gpu_name = "cpu"

print("Device:", DEVICE)
print("GPU:", gpu_name)

EPOCHS = 50
BATCH = 8
LR = 5e-5
PATIENCE = 10
DET_CONF = 0.25

CLASSIFIERS = [
    "vit_b16_imagenet",
]

Device: 0
GPU: NVIDIA RTX 2000 Ada Generation


## 4) Encontrar o melhor peso do Path A

A escolha do detector YOLO define a primeira etapa do pipeline Path B3. Como o B3 avalia classificação a partir de objetos detectados, a qualidade do detector influencia diretamente quais crops chegam ao ViT.

A célula percorre diferentes diretórios de runs do Path A, procura checkpoints `weights/best.pt` e lê métricas de `results.csv` ou, se necessário, de `metrics.json`. Os candidatos são ranqueados priorizando `mAP50-95`; se essa métrica não existir, o ranking usa `mAP50`. O melhor checkpoint encontrado é salvo em `DETECTOR_WEIGHTS` para ser usado nas próximas etapas.

In [8]:
PATH_A_RUN_DIRS = [
    WORKSPACE / "runs" / "path_A",
    WORKSPACE / "runs" / "path_A_5cls",
    WORKSPACE / "runs" / "path_A_refined_head",
]

def read_yolo_results(run_dir: Path):
    """
    Lê métricas de uma pasta de treino YOLO.
    Espera estrutura:
      run_dir/
        weights/best.pt
        results.csv
        args.yaml
    """
    best_pt = run_dir / "weights" / "best.pt"
    results_csv = run_dir / "results.csv"
    metrics_json = run_dir / "metrics.json"

    if not best_pt.exists():
        return None

    row = {
        "group": run_dir.parent.name,
        "model": run_dir.name,
        "run_dir": run_dir,
        "best_pt": best_pt,
        "mAP50_95": None,
        "mAP50": None,
        "precision": None,
        "recall": None,
        "source": None,
    }

    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]

        # Melhor época por mAP50-95, se existir
        map95_col = "metrics/mAP50-95(B)"
        map50_col = "metrics/mAP50(B)"
        precision_col = "metrics/precision(B)"
        recall_col = "metrics/recall(B)"

        if map95_col in df.columns:
            best_idx = df[map95_col].idxmax()
        elif map50_col in df.columns:
            best_idx = df[map50_col].idxmax()
        else:
            best_idx = df.index[-1]

        best = df.loc[best_idx]

        row["mAP50_95"] = float(best[map95_col]) if map95_col in df.columns else None
        row["mAP50"] = float(best[map50_col]) if map50_col in df.columns else None
        row["precision"] = float(best[precision_col]) if precision_col in df.columns else None
        row["recall"] = float(best[recall_col]) if recall_col in df.columns else None
        row["source"] = "results.csv"
        return row

    # Fallback para metrics.json, se existir
    if metrics_json.exists():
        with open(metrics_json, "r") as f:
            m = json.load(f)

        row["mAP50_95"] = m.get("mAP50_95")
        row["mAP50"] = m.get("mAP50")
        row["precision"] = m.get("precision")
        row["recall"] = m.get("recall")
        row["source"] = "metrics.json"
        return row

    # Tem best.pt, mas sem métrica
    row["source"] = "weights_only"
    return row


records = []

for base_dir in PATH_A_RUN_DIRS:
    if not base_dir.exists():
        print(f"[warn] Pasta não encontrada: {base_dir}")
        continue

    for run_dir in sorted(base_dir.iterdir()):
        if not run_dir.is_dir():
            continue

        rec = read_yolo_results(run_dir)
        if rec is not None:
            records.append(rec)

df_detectors = pd.DataFrame(records)

if df_detectors.empty:
    raise FileNotFoundError(
        "Nenhum detector com weights/best.pt foi encontrado em: "
        + ", ".join(str(p) for p in PATH_A_RUN_DIRS)
    )

# Ordena pelo melhor critério disponível
df_ranked = df_detectors.copy()
df_ranked["rank_score"] = df_ranked["mAP50_95"].fillna(df_ranked["mAP50"]).fillna(-1)

df_ranked = df_ranked.sort_values(
    by=["rank_score", "mAP50", "precision", "recall"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)

display_cols = [
    "group", "model", "mAP50_95", "mAP50", "precision", "recall", "source", "best_pt"
]

display(df_ranked[display_cols])

best_detector = df_ranked.iloc[0]
DETECTOR_WEIGHTS = Path(best_detector["best_pt"])

print("Melhor detector encontrado:")
print("Grupo :", best_detector["group"])
print("Modelo:", best_detector["model"])
print("mAP50-95:", best_detector["mAP50_95"])
print("mAP50:", best_detector["mAP50"])
print("Pesos:", DETECTOR_WEIGHTS)

if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(f"Detector não encontrado: {DETECTOR_WEIGHTS}")

,group,model,mAP50_95,mAP50,precision,recall,source,best_pt
0,path_A_5cls,yolov11m_o2o,0.49421,0.71055,0.79748,0.64537,results.csv,/workspace/runs/path_A_5cls/yolov11m_o2o/weigh...
1,path_A_5cls,yolov11m,0.45791,0.66802,0.76878,0.61979,results.csv,/workspace/runs/path_A_5cls/yolov11m/weights/b...
2,path_A_5cls,yolov8m,0.45466,0.67461,0.81395,0.59336,results.csv,/workspace/runs/path_A_5cls/yolov8m/weights/be...
3,path_A,yolov11m,0.44551,0.64990,0.70430,0.58665,results.csv,/workspace/runs/path_A/yolov11m/weights/best.pt
4,path_A,yolov11m_o2o,0.44040,0.64164,0.68838,0.59821,results.csv,/workspace/runs/path_A/yolov11m_o2o/weights/be...
5,path_A,yolov10m,0.39211,0.57796,0.63463,0.53399,results.csv,/workspace/runs/path_A/yolov10m/weights/best.pt
6,path_A,yolov9s,0.38005,0.57062,0.65521,0.51124,results.csv,/workspace/runs/path_A/yolov9s/weights/best.pt
7,path_A_refined_head,yolov10n,0.04614,0.11337,0.18439,0.16914,results.csv,/workspace/runs/path_A_refined_head/yolov10n/w...
8,path_A,yolov8m,NaN,NaN,NaN,NaN,weights_only,/workspace/runs/path_A/yolov8m/weights/best.pt


Melhor detector encontrado:
Grupo : path_A_5cls
Modelo: yolov11m_o2o
mAP50-95: 0.49421
mAP50: 0.71055
Pesos: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt


## 5) Conferir estrutura do Path B

A checagem percorre os splits de treinamento, validação e teste para confirmar que o dataset processado possui a estrutura esperada. O script do Path B espera encontrar, para cada split, a pasta `path_B` com `images`, `labels` e `crops`.

Mesmo quando o treino usa `--use_yolo_crops`, as imagens e labels continuam necessárias: o YOLO gera detecções nas imagens, e as labels são usadas para associar cada box detectada à classe correta durante a construção dos crops supervisionados.

In [6]:
for split in ["train", "val", "test"]:
    path_b_dir = PROCESSED_DIR / split / "path_B"
    print(split, path_b_dir, "->", path_b_dir.exists())

    for sub in ["images", "labels", "crops"]:
        p = path_b_dir / sub
        print("  ", sub, "->", p.exists())

train /workspace/processed_5cls/train/path_B -> True
   images -> True
   labels -> True
   crops -> True
val /workspace/processed_5cls/val/path_B -> True
   images -> True
   labels -> True
   crops -> True
test /workspace/processed_5cls/test/path_B -> True
   images -> True
   labels -> True
   crops -> True


## 6) Treino Path B3 — ViT-B/16 fine-tuning (TTA no YOLO)

O fine-tuning do B3 usa crops produzidos pelo detector YOLO selecionado, com TTA ativado na etapa de detecção. Nesta etapa, apenas o classificador é treinado; o detector entra como modelo congelado para localizar objetos e gerar os recortes.

O comando usa `--use_yolo_crops` e `--tta`, então os crops vêm das detecções com TTA. Os crops são salvos em cache para acelerar reexecuções, e o script registra `history.csv`, `metrics.json`, `weights/best.pt` e matriz de confusão dentro da run do classificador.

In [16]:
print("\nParametros:")

RUNS_PATH_B_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python executable: {sys.executable}")
print(f"Script: {TRAIN_PATH_B_SCRIPT}")
print(f"Detector weights: {DETECTOR_WEIGHTS}")
print(f"Crops dir: {PROCESSED_DIR}")
print(f"Output dir: {RUNS_PATH_B_DIR}")
print(f"Classifiers: {CLASSIFIERS}")
print(f"Epochs: {EPOCHS}")
print(f"Batch: {BATCH}")
print(f"Learning rate: {LR}")
print(f"Patience: {PATIENCE}")
print(f"Device: {DEVICE}")
print("Use YOLO crops: True")
print(f"Detection confidence: {DET_CONF}")
print(f"Crop cache dir: {RUNS_PATH_B_DIR / 'crop_cache_tta'}")


Parametros:
Python executable: /workspace/.venv/bin/python
Script: /workspace/TrashScan/train/paths/train_path_B.py
Detector weights: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
Crops dir: /workspace/processed_5cls
Output dir: /workspace/runs/path_B_tta_wbf
Classifiers: ['vit_b16_imagenet']
Epochs: 50
Batch: 8
Learning rate: 5e-05
Patience: 10
Device: 0
Use YOLO crops: True
Detection confidence: 0.25
Crop cache dir: /workspace/runs/path_B_tta_wbf/crop_cache_tta


In [17]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--classifiers", *CLASSIFIERS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--lr", str(LR),
    "--patience", str(PATIENCE),
    "--device", str(DEVICE),
    "--use_yolo_crops",
    "--det_conf", str(DET_CONF),
    "--use_tta_wbf",
    "--tta_scales", "512", "640", "768",
    "--tta_flip",
    "--tta_wbf_iou", "0.55",
    "--tta_skip_box_thr", "0.001",
    "--crop_cache_dir", str(RUNS_PATH_B_DIR / "crop_cache_tta_wbf"),
])

$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B_tta_wbf --classifiers vit_b16_imagenet --epochs 50 --batch 8 --lr 5e-05 --patience 10 --device 0 --use_yolo_crops --det_conf 0.25 --use_tta_wbf --tta_scales 512 640 768 --tta_flip --tta_wbf_iou 0.55 --tta_skip_box_thr 0.001 --crop_cache_dir /workspace/runs/path_B_tta_wbf/crop_cache_tta_wbf


Device : cuda:0
GPU    : NVIDIA RTX 2000 Ada Generation  VRAM: 16.9GB
Class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

  Mode: YOLO on-the-fly crop extraction
  YOLO TTA+WBF: True
  TTA scales: [512, 640, 768]
  TTA flip: True
  TTA WBF IoU: 0.55
  TTA skip box thr: 0.001
  [train] Loading YOLOCropDataset from cache: /workspace/runs/path_B_tta_wbf/crop_cache_tta_wbf/train
  [train] 11346 cached crops loaded
  [val] Loading YOLOCropDataset from cache: /workspace/runs/path_B_tta_wbf/crop_cache_tta_wbf/val
  [val] 2254 cached crops loaded
  [test] Loading YOLOCropDataset from cache: /workspace/runs/path_B_tta_wbf/crop_cache_tta_wbf/test
  [test] 2221 cached crops loaded

────────────────────────────────────────────────────────────
  Classifier : vit_b16_imagenet
────────────────────────────────────────────────────────────


  Built vit_b16_imagenet (vit_base_patch16_224.augreg_in21k_ft_in1k)  pretrained=True  85.8M params


  Ep   1/50 | train loss=1.0368 acc=0.6660 | val loss=0.6179 acc=0.7919 f1=0.6898


  Ep   2/50 | train loss=0.6594 acc=0.7847 | val loss=0.4564 acc=0.8438 f1=0.7821


...

  Ep  42/50 | train loss=0.0754 acc=0.9611 | val loss=0.8539 acc=0.9255 f1=0.9206


  Ep  43/50 | train loss=0.0699 acc=0.9620 | val loss=0.9481 acc=0.9232 f1=0.9177
  Early stopping at epoch 43 (best epoch 33, val_acc=0.9264)


  Test: 100%|██████████| 278/278 [00:15<00:00, 17.50it/s]



  [vit_b16_imagenet]  accuracy=0.9293  f1=0.9174  latency=8.59ms

  [vit_b16_imagenet]  best_epoch=33  val_acc=0.9264  test_acc=0.9293  f1=0.9174

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_b16_imagenet    0.9293     0.8987  0.9407 0.9174      8.5950      0.9552    0.9152    0.8324    0.9487    0.9656
vit_l16_imagenet    0.9244     0.8776  0.9330 0.9020     31.2870      0.9760    0.9439    0.8603    0.9536    0.9737


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B_tta_wbf', '--classifiers', 'vit_b16_imagenet', '--epochs', '50', '--batch', '8', '--lr', '5e-05', '--patience', '10', '--device', '0', '--use_yolo_crops', '--det_conf', '0.25', '--use_tta_wbf', '--tta_scales', '512', '640', '768', '--tta_flip', '--tta_wbf_iou', '0.55', '--tta_skip_box_thr', '0.001', '--crop_cache_dir', '/workspace/runs/path_B_tta_wbf/crop_cache_tta_wbf'], returncode=0)

In [35]:
run_dir = Path("/workspace/runs/path_B_5cls/vit_b16_imagenet")

print("best.pt:", (run_dir / "weights" / "best.pt").exists())
print("history.csv:", (run_dir / "history.csv").exists())
print("metrics.json:", (run_dir / "metrics.json").exists())

if (run_dir / "metrics.json").exists():
    print(json.loads((run_dir / "metrics.json").read_text()))

if (run_dir / "history.csv").exists():
    hist = pd.read_csv(run_dir / "history.csv")
    display(hist.tail())

best.pt: True
history.csv: True
metrics.json: True
{'classifier': 'vit_b16_imagenet', 'accuracy': 0.91537, 'precision': 0.88258, 'recall': 0.92785, 'f1': 0.90321, 'latency_ms': 11.714, 'AP_plastic': 0.9659, 'AP_paper': 0.93342, 'AP_metal': 0.85586, 'AP_glass': 0.95129, 'AP_other': 0.96765}


,epoch,train_loss,train_acc,val_loss,val_acc,val_f1,lr
23,24,0.17484,0.93580,0.42372,0.91185,0.89792,0.000043
24,25,0.16462,0.93886,0.41430,0.90104,0.88838,0.000043
25,26,0.15537,0.93927,0.47029,0.91310,0.90428,0.000042
26,27,0.15243,0.94217,0.41621,0.90977,0.90541,0.000042
27,28,0.15772,0.93894,0.41887,0.90686,0.89755,0.000041


## 7) Resumo do treino

O resumo consolida os resultados já salvos pelo script do Path B. A opção `--summarize` percorre as runs disponíveis em `/workspace/runs/path_B_5cls` e imprime uma visão resumida das métricas do classificador treinado no B3 com TTA.

Também há redefinição explícita de alguns caminhos para facilitar reexecuções isoladas a partir daqui. Atenção: `DETECTOR_WEIGHTS` passa a apontar manualmente para `/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt`.

In [36]:
def run_cmd(cmd, cwd="/workspace", env=None, shell=False):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    if shell:
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)
   
TRAIN_PATH_B_SCRIPT = Path("/workspace/TrashScan/train/paths/train_path_B.py")
RUNS_PATH_B_DIR = Path("/workspace/runs/path_B_5cls")
DETECTOR_WEIGHTS = Path("/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt")
PROCESSED_DIR = Path("/workspace/processed_5cls")

In [37]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--summarize",
])

$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B_5cls --summarize

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_b16_imagenet    0.9154     0.8826  0.9278 0.9032     11.7140      0.9659    0.9334    0.8559    0.9513    0.9677


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B_5cls', '--summarize'], returncode=0)

## 8) Ver métricas do ViT-B/16 (TTA)

A inspeção final olha para os artefatos da run `vit_b16_imagenet` dentro de `/workspace/runs/path_B_5cls`. O histórico de treino é lido para encontrar as melhores épocas por acurácia de validação e F1 macro de validação, e essas informações são combinadas com as métricas de teste salvas em `metrics.json`.

As principais leituras desta seção são: quantas épocas foram treinadas, se `weights/best.pt` foi salvo, qual foi a melhor validação, qual foi o desempenho no teste e se a matriz de confusão foi gerada. As métricas macro são importantes porque o dataset de resíduos tende a ser desbalanceado; elas reduzem o risco de interpretar apenas a performance nas classes mais frequentes.

In [38]:
RUN_DIR = Path("/workspace/runs/path_B_5cls/vit_b16_imagenet")

history_path = RUN_DIR / "history.csv"
metrics_path = RUN_DIR / "metrics.json"
best_weights_path = RUN_DIR / "weights" / "best.pt"
cm_path = RUN_DIR / "confusion_matrix_vit_b16_imagenet.png"

print("Run dir:", RUN_DIR)
print("best.pt existe?", best_weights_path.exists())
print("history.csv existe?", history_path.exists())
print("metrics.json existe?", metrics_path.exists())
print("confusion matrix existe?", cm_path.exists())

if not history_path.exists():
    raise FileNotFoundError(f"history.csv não encontrado: {history_path}")

hist = pd.read_csv(history_path)

best_val_acc_idx = hist["val_acc"].idxmax()
best_val_f1_idx = hist["val_f1"].idxmax()

best_val_acc_row = hist.loc[best_val_acc_idx]
best_val_f1_row = hist.loc[best_val_f1_idx]

summary = {
    "run_dir": str(RUN_DIR),
    "best_weights": str(best_weights_path),
    "best_weights_exists": best_weights_path.exists(),

    "epochs_trained": int(hist["epoch"].max()),

    "best_epoch_by_val_acc": int(best_val_acc_row["epoch"]),
    "best_val_acc": float(best_val_acc_row["val_acc"]),
    "best_val_acc_val_f1": float(best_val_acc_row["val_f1"]),
    "best_val_acc_train_acc": float(best_val_acc_row["train_acc"]),
    "best_val_acc_train_loss": float(best_val_acc_row["train_loss"]),
    "best_val_acc_val_loss": float(best_val_acc_row["val_loss"]),

    "best_epoch_by_val_f1": int(best_val_f1_row["epoch"]),
    "best_val_f1": float(best_val_f1_row["val_f1"]),
    "best_val_f1_val_acc": float(best_val_f1_row["val_acc"]),
}

if metrics_path.exists():
    with open(metrics_path, "r") as f:
        test_metrics = json.load(f)

    summary.update({
        "test_accuracy": test_metrics.get("accuracy"),
        "test_precision_macro": test_metrics.get("precision"),
        "test_recall_macro": test_metrics.get("recall"),
        "test_f1_macro": test_metrics.get("f1"),
        "latency_ms": test_metrics.get("latency_ms"),
        "AP_plastic": test_metrics.get("AP_plastic"),
        "AP_paper": test_metrics.get("AP_paper"),
        "AP_metal": test_metrics.get("AP_metal"),
        "AP_glass": test_metrics.get("AP_glass"),
        "AP_other": test_metrics.get("AP_other"),
    })
else:
    print("\n⚠️ metrics.json não foi encontrado. Talvez o erro no MLflow tenha ocorrido antes de salvar as métricas.")

summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
display(summary_df)

if metrics_path.exists():
    print("\nMétricas de teste:")
    display(pd.DataFrame([test_metrics]))

Run dir: /workspace/runs/path_B_5cls/vit_b16_imagenet
best.pt existe? True
history.csv existe? True
metrics.json existe? True
confusion matrix existe? True


,value
run_dir,/workspace/runs/path_B_5cls/vit_b16_imagenet
best_weights,/workspace/runs/path_B_5cls/vit_b16_imagenet/w...
best_weights_exists,True
epochs_trained,28
best_epoch_by_val_acc,18
best_val_acc,0.91518
best_val_acc_val_f1,0.90548
best_val_acc_train_acc,0.93241
best_val_acc_train_loss,0.19051
best_val_acc_val_loss,0.34693



Métricas de teste:


,classifier,accuracy,precision,recall,f1,latency_ms,AP_plastic,AP_paper,AP_metal,AP_glass,AP_other
0,vit_b16_imagenet,0.91537,0.88258,0.92785,0.90321,11.714,0.9659,0.93342,0.85586,0.95129,0.96765


## 9) Avaliação combinada detector + classificador

A avaliação combinada mede o comportamento do sistema completo em modo próximo ao uso real. O script `evaluate_path_B_combined.py` recebe o detector YOLO, a pasta de classificadores do Path B (TTA) e o YAML do dataset para executar a cadeia completa: detectar objetos, recortar regiões, classificar cada crop com o B3 e salvar os resultados finais.

Os parâmetros `det_conf=0.001` e `det_iou=0.6` controlam a etapa de detecção durante a avaliação combinada. O resultado é salvo em `/workspace/results_path_B/b3_5cls_tta`, separando a avaliação final dos artefatos brutos de treino em `/workspace/runs/path_B_5cls`.

In [20]:

EVAL_PATH_B_COMBINED_SCRIPT = Path("/workspace/TrashScan/eval/evaluate_path_B_combined.py")

DETECTOR_WEIGHTS = Path("/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt")
CLASSIFIER_DIR = Path("/workspace/runs/path_B_tta_wbf")

DATA_YAML = Path("/workspace/processed_5cls/dataset_path_A.yaml")

OUTPUT_DIR = Path("/workspace/results_path_B/b3_5cls_tta_wbf")

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--classifier_dir", str(CLASSIFIER_DIR),
    "--classifiers", "vit_b16_imagenet",
    "--data_yaml", str(DATA_YAML),
    "--output", str(OUTPUT_DIR),
    "--device", "0",
    "--imgsz", "640",
    "--det_conf", "0.001",
    "--det_iou", "0.6",
    "--use_tta_wbf",
    "--tta_scales", "512", "640", "768",
    "--tta_flip",
    "--tta_wbf_iou", "0.55",
    "--tta_skip_box_thr", "0.001",
]

print("$", " ".join(shlex.quote(str(x)) for x in cmd))

subprocess.run(
    [str(x) for x in cmd],
    cwd="/workspace",
    check=True,
)

$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_path_B_combined.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --classifier_dir /workspace/runs/path_B_tta_wbf --classifiers vit_b16_imagenet --data_yaml /workspace/processed_5cls/dataset_path_A.yaml --output /workspace/results_path_B/b3_5cls_tta_wbf --device 0 --imgsz 640 --det_conf 0.001 --det_iou 0.6 --use_tta_wbf --tta_scales 512 640 768 --tta_flip --tta_wbf_iou 0.55 --tta_skip_box_thr 0.001


Device: cuda:0
Test set: /workspace/processed_5cls/test/path_A/images

Loading detector: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt

────────────────────────────────────────────────────────────
  Evaluating: vit_b16_imagenet
────────────────────────────────────────────────────────────
  Loaded vit_b16_imagenet: 85.8M params from best.pt

  Running combined inference on 1392 test images...
  TTA+WBF: True
  TTA scales: [512, 640, 768]
  TTA flip: True
  TTA WBF IoU: 0.55
  TTA skip box thr: 0.001


  vit_b16_imagenet: 100%|██████████| 1392/1392 [04:36<00:00,  5.03it/s]



  [vit_b16_imagenet]  mAP50=0.6721  mAP50-95=0.4628  fps=5.0
    AP50 plastic : 0.7071  (n_gt=1313)
    AP50 paper   : 0.6744  (n_gt=280)
    AP50 metal   : 0.6057  (n_gt=265)
    AP50 glass   : 0.6682  (n_gt=88)
    AP50 other   : 0.7051  (n_gt=1263)
  Saved: /workspace/results_path_B/b3_5cls_tta_wbf/individual/B_vit_b16_imagenet_combined.json

PATH B COMBINED — FINAL COMPARISON
Classifier                  mAP50  mAP50-95    FPS
──────────────────────────────────────────────────
  vit_b16_imagenet         0.6721    0.4628    5.0

Summary saved: /workspace/results_path_B/b3_5cls_tta_wbf/path_B_combined_summary.json


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_path_B_combined.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--classifier_dir', '/workspace/runs/path_B_tta_wbf', '--classifiers', 'vit_b16_imagenet', '--data_yaml', '/workspace/processed_5cls/dataset_path_A.yaml', '--output', '/workspace/results_path_B/b3_5cls_tta_wbf', '--device', '0', '--imgsz', '640', '--det_conf', '0.001', '--det_iou', '0.6', '--use_tta_wbf', '--tta_scales', '512', '640', '768', '--tta_flip', '--tta_wbf_iou', '0.55', '--tta_skip_box_thr', '0.001'], returncode=0)